In [ ]:
import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f'Seed set: {seed}')
set_seed()
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

In [ ]:
class Config:
    # ── Paths ────────────────────────────────────────────────────
    data_path    = '/home/user/Downloads/PEMS08.npz'
    adj_csv_path = '/home/user/Downloads/PEMS08.csv'
    num_nodes    = 170
    in_features  = 3
    seq_len      = 12
    pred_len     = 12
    feature_idx  = 0
    train_ratio  = 0.7
    val_ratio    = 0.1

    # ── Model ────────────────────────────────────────────────────
    d_model      = 96
    d_skip       = 384
    d_end        = 256
    d_time       = 64
    n_layers     = 12
    kernel_size  = 4
    adp_emb      = 20
    gcn_order    = 3
    n_supports   = 3
    dropout      = 0.1
    n_heads_maf  = 4
    n_stf_layers = 2
    n_heads_stf  = 4
    huber_delta  = 0.2

    # ── Training ─────────────────────────────────────────────────
    batch_size   = 48
    lr           = 1e-3
    epochs       = 200   # FIX: was 150
    patience     = 50    # FIX: was 30
    weight_decay = 1e-4
    best_path    = 'mdgwnet_best.pt'

cfg    = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'MD-GWNet v21 (fixed) | d={cfg.d_model} | layers={cfg.n_layers} | {device}')

In [ ]:
HOUR_OFFSET = 12
DAY_OFFSET  = 288

def load_pems08(cfg):
    import pandas as pd, os
    raw  = np.load(cfg.data_path)
    data = raw['data'].astype(np.float32)
    T, N, F = data.shape
    print(f'Shape: {data.shape}')
    mean_np = data.mean(axis=0)
    std_np  = data.std(axis=0) + 1e-8
    data_norm = (data - mean_np[None]) / std_np[None]

    A_dist = None
    if os.path.exists(cfg.adj_csv_path):
        try:
            df_adj = pd.read_csv(cfg.adj_csv_path, header=None, skiprows=1)
            df_adj.columns = ['from', 'to', 'cost']
            A_raw  = np.zeros((N, N), dtype=np.float32)
            for _, row in df_adj.iterrows():
                i, j, c = int(row['from']), int(row['to']), float(row['cost'])
                if i < N and j < N:
                    A_raw[i,j] = c; A_raw[j,i] = c
            nonzero = A_raw[A_raw > 0]
            sigma   = nonzero.std() if len(nonzero) > 0 else 1.0
            A_dist  = np.exp(-(A_raw**2) / (sigma**2 + 1e-8))
            np.fill_diagonal(A_dist, 0.0)
            A_dist  = A_dist / (A_dist.sum(1, keepdims=True) + 1e-8)
            print(f'Adj: {(A_dist>0).sum()} nnz, avg_deg={(A_dist>0).sum()/N:.1f}')
        except Exception as e:
            print(f'Adj CSV failed: {e}')
            A_dist = None

    if A_dist is None:
        A_dist = np.eye(N, dtype=np.float32)
        print('WARNING: using identity adjacency')
    return data_norm, mean_np, std_np, A_dist


class TrafficDataset3Stream(Dataset):
    def __init__(self, data, seq_len, pred_len, feature_idx,
                 split_start=0, split_end=None):
        self.data     = data
        self.seq_len  = seq_len
        self.pred_len = pred_len
        self.feat_idx = feature_idx
        T = len(data)
        split_end = split_end or T
        eff_start = max(split_start, DAY_OFFSET)
        last_i    = split_end - seq_len - pred_len + 1
        self.indices = list(range(eff_start, last_i))

    def __len__(self): return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        S, P = self.seq_len, self.pred_len
        rec  = self.data[i              : i+S             ].copy()
        hour = self.data[i-HOUR_OFFSET  : i-HOUR_OFFSET+S ].copy()
        day  = self.data[i-DAY_OFFSET   : i-DAY_OFFSET+S  ].copy()
        y    = self.data[i+S : i+S+P,   :, self.feat_idx  ].copy()
        tod  = np.array([(i+t) % 288      for t in range(S)], dtype=np.int64)
        dow  = np.array([((i+t)//288) % 7 for t in range(S)], dtype=np.int64)
        return (torch.from_numpy(rec),  torch.from_numpy(hour),
                torch.from_numpy(day),  torch.from_numpy(y),
                torch.from_numpy(tod),  torch.from_numpy(dow))


def build_dataloaders(cfg):
    set_seed()
    data, mean_np, std_np, A_dist = load_pems08(cfg)
    T  = len(data)
    t1 = int(T * cfg.train_ratio)
    t2 = int(T * (cfg.train_ratio + cfg.val_ratio))
    kw    = dict(batch_size=cfg.batch_size, num_workers=2, pin_memory=True)
    ds_kw = dict(data=data, seq_len=cfg.seq_len, pred_len=cfg.pred_len,
                 feature_idx=cfg.feature_idx)
    dl_tr = DataLoader(TrafficDataset3Stream(**ds_kw, split_start=0,  split_end=t1), shuffle=True,  **kw)
    dl_va = DataLoader(TrafficDataset3Stream(**ds_kw, split_start=t1, split_end=t2), shuffle=False, **kw)
    dl_te = DataLoader(TrafficDataset3Stream(**ds_kw, split_start=t2, split_end=T),  shuffle=False, **kw)
    print(f'Train={len(dl_tr.dataset)} | Val={len(dl_va.dataset)} | Test={len(dl_te.dataset)}')
    return dl_tr, dl_va, dl_te, mean_np, std_np, A_dist

print('3-stream dataset ready.')

In [ ]:
class DiffusionGCN(nn.Module):
    def __init__(self, d_in, d_out, n_supports=3, order=2, dropout=0.1):
        super().__init__()
        self.mlp   = nn.Linear(d_in * (1 + n_supports * order), d_out)
        self.drop  = nn.Dropout(dropout)
        self.order = order
    def forward(self, x, supports):  # x: (B*S, N, d)
        hs = [x]
        for A in supports:
            h = x
            for _ in range(self.order):
                h = torch.einsum('nm,bmd->bnd', A, h)
                hs.append(h)
        return self.drop(self.mlp(torch.cat(hs, dim=-1)))


class WaveBlock(nn.Module):
    def __init__(self, d_model, d_skip, kernel_size, dilation,
                 n_supports, gcn_order, dropout):
        super().__init__()
        self.dilation    = dilation
        self.kernel_size = kernel_size
        conv_kw = dict(kernel_size=(1, kernel_size), dilation=(1, dilation))
        self.filter_conv = nn.Conv2d(d_model, d_model, **conv_kw)
        self.gate_conv   = nn.Conv2d(d_model, d_model, **conv_kw)
        self.gcn         = DiffusionGCN(d_model, d_model, n_supports, gcn_order, dropout)
        self.bn          = nn.BatchNorm2d(d_model)
        self.drop        = nn.Dropout(dropout)
        self.skip_conv   = nn.Conv2d(d_model, d_skip,  (1,1))
        self.res_conv    = nn.Conv2d(d_model, d_model, (1,1))

    def forward(self, x, supports):  # x: (B, d, N, S)
        residual = x
        pad = (self.kernel_size - 1) * self.dilation
        x_pad = F.pad(x, [pad, 0])
        f = torch.tanh   (self.filter_conv(x_pad))
        g = torch.sigmoid(self.gate_conv  (x_pad))
        x = self.drop(f * g)
        B, d, N, S = x.shape
        xg = x.permute(0,3,2,1).reshape(B*S, N, d)  # (B*S, N, d)
        xg = self.gcn(xg, supports)
        x  = xg.reshape(B,S,N,d).permute(0,3,2,1)   # (B, d, N, S)
        x  = self.bn(x)
        skip = self.skip_conv(x)
        x    = self.res_conv(x) + residual
        return x, skip


class MultiPeriodFusion(nn.Module):
    """
    Cross-attention: recent (query) attends to hourly+daily (key/value).
    BUG FIX: final permute was (0,3,2,1) which gave (B,d,S,N) instead of
    (B,d,N,S), swapping N and S → caused einsum crash in DiffusionGCN.
    FIXED to permute(0,3,1,2).
    """
    def __init__(self, d_model, n_heads=4, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.d_head  = d_model // n_heads
        assert d_model % n_heads == 0
        self.q_proj   = nn.Linear(d_model, d_model)
        self.k_proj   = nn.Linear(d_model * 2, d_model)
        self.v_proj   = nn.Linear(d_model * 2, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.drop     = nn.Dropout(dropout)
        self.norm     = nn.LayerNorm(d_model)

    def forward(self, h_rec, h_hour, h_day):  # each: (B, d, N, S)
        B, d, N, S = h_rec.shape
        nh, dh     = self.n_heads, self.d_head

        def flatten(t):  # (B, d, N, S) → (B*N, S, d)
            return t.permute(0,2,3,1).reshape(B*N, S, d)

        rec  = flatten(h_rec)
        hour = flatten(h_hour)
        day  = flatten(h_day)
        kv   = torch.cat([hour, day], dim=-1)           # (B*N, S, 2d)

        Q = self.q_proj(rec ).reshape(B*N, S, nh, dh).transpose(1,2)
        K = self.k_proj(kv  ).reshape(B*N, S, nh, dh).transpose(1,2)
        V = self.v_proj(kv  ).reshape(B*N, S, nh, dh).transpose(1,2)

        attn = F.softmax(Q @ K.transpose(-2,-1) / dh**0.5, dim=-1)
        attn = self.drop(attn)
        out  = (attn @ V).transpose(1,2).reshape(B*N, S, d)
        out  = self.out_proj(out)
        out  = self.norm(out + rec)                    # (B*N, S, d)

        # ═══════════════════════════════════════════════════════════
        # FIX: was permute(0,3,2,1) → gave (B, d, S, N) WRONG
        # CORRECT: permute(0,3,1,2) → gives (B, d, N, S)
        # Derivation: out.reshape(B,N,S,d), permute(0,3,1,2):
        #   new_dim0=old0=B, new_dim1=old3=d, new_dim2=old1=N, new_dim3=old2=S
        #   → (B, d, N, S) ✓
        # ═══════════════════════════════════════════════════════════
        return out.reshape(B, N, S, d).permute(0, 3, 1, 2)  # (B, d, N, S)


class SpatialTransformer(nn.Module):
    def __init__(self, d, n_heads, dropout=0.1):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)
        self.ff    = nn.Sequential(nn.Linear(d, d*2), nn.GELU(),
                                   nn.Dropout(dropout), nn.Linear(d*2, d))
        self.norm1 = nn.LayerNorm(d)
        self.norm2 = nn.LayerNorm(d)

    def forward(self, x):  # (B, d, N, S)
        B, d, N, S = x.shape
        xt = x.permute(0,3,2,1).reshape(B*S, N, d)  # (B*S, N, d)
        a, _ = self.attn(xt, xt, xt)
        xt = self.norm1(xt + a)
        xt = self.norm2(xt + self.ff(xt))
        return xt.reshape(B, S, N, d).permute(0,3,2,1)  # (B, d, N, S)


class TemporalTransformer(nn.Module):
    def __init__(self, d, n_heads, dropout=0.1):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)
        self.ff    = nn.Sequential(nn.Linear(d, d*2), nn.GELU(),
                                   nn.Dropout(dropout), nn.Linear(d*2, d))
        self.norm1 = nn.LayerNorm(d)
        self.norm2 = nn.LayerNorm(d)

    def forward(self, x):  # (B, d, N, S)
        B, d, N, S = x.shape
        xn = x.permute(0,2,3,1).reshape(B*N, S, d)  # (B*N, S, d)
        a, _ = self.attn(xn, xn, xn)
        xn = self.norm1(xn + a)
        xn = self.norm2(xn + self.ff(xn))
        return xn.reshape(B, N, S, d).permute(0,3,2,1)  # (B, d, N, S)


class STFormer(nn.Module):
    def __init__(self, d, n_layers=2, n_heads=4, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.ModuleList([
                SpatialTransformer(d, n_heads, dropout),
                TemporalTransformer(d, n_heads, dropout)
            ]) for _ in range(n_layers)
        ])
    def forward(self, x):
        for sp, tp in self.layers:
            x = tp(sp(x))
        return x

print('All blocks defined (MultiPeriodFusion permute bug fixed).')

In [ ]:
class MDGWNet(nn.Module):
    def __init__(self, cfg, A_fwd, A_bwd):
        super().__init__()
        N, d = cfg.num_nodes, cfg.d_model

        self.sc_rec  = nn.Conv2d(cfg.in_features, d, (1,1))
        self.sc_hour = nn.Conv2d(cfg.in_features, d, (1,1))
        self.sc_day  = nn.Conv2d(cfg.in_features, d, (1,1))

        self.emb_tod   = nn.Embedding(288, cfg.d_time)
        self.emb_dow   = nn.Embedding(7,   cfg.d_time)
        self.time_proj = nn.Linear(cfg.d_time * 2, d)
        self.node_emb  = nn.Parameter(torch.randn(1, d, N, 1) * 0.01)

        self.E1 = nn.Parameter(torch.randn(N, cfg.adp_emb))
        self.E2 = nn.Parameter(torch.randn(cfg.adp_emb, N))

        self.register_buffer('A_fwd', A_fwd)
        self.register_buffer('A_bwd', A_bwd)

        self.maf = MultiPeriodFusion(d, n_heads=cfg.n_heads_maf, dropout=cfg.dropout)

        dilations = [2**i for i in range(4)] * (cfg.n_layers // 4)
        self.blocks = nn.ModuleList([
            WaveBlock(d, cfg.d_skip, cfg.kernel_size, dil,
                      cfg.n_supports, cfg.gcn_order, cfg.dropout)
            for dil in dilations
        ])

        self.skip_proj = nn.Conv2d(cfg.d_skip, cfg.d_end, (1,1))
        self.stformer  = STFormer(cfg.d_end, n_layers=cfg.n_stf_layers,
                                  n_heads=cfg.n_heads_stf, dropout=cfg.dropout)
        self.out_conv  = nn.Sequential(
            nn.Conv2d(cfg.d_end, cfg.d_end, (1,1)), nn.ReLU(),
            nn.Conv2d(cfg.d_end, cfg.pred_len, (1,1))
        )

    def _supports(self):
        A_adp = F.softmax(F.relu(self.E1 @ self.E2), dim=-1)
        return [self.A_fwd, self.A_bwd, A_adp]

    def forward(self, x_rec, x_hour, x_day, tod=None, dow=None):
        def to4d(x): return x.permute(0,3,2,1)  # (B,S,N,F) → (B,F,N,S)

        h_rec  = self.sc_rec (to4d(x_rec ))  + self.node_emb
        h_hour = self.sc_hour(to4d(x_hour))
        h_day  = self.sc_day (to4d(x_day ))

        if tod is not None and dow is not None:
            te = self.time_proj(torch.cat([self.emb_tod(tod),
                                           self.emb_dow(dow)], dim=-1))  # (B,S,d)
            h_rec = h_rec + te.permute(0,2,1).unsqueeze(2)               # (B,d,1,S) broadcast

        x = self.maf(h_rec, h_hour, h_day)   # (B, d, N, S) — BUG FIXED

        sup = self._supports()
        skip_acc = 0
        for blk in self.blocks:
            x, skip = blk(x, sup)
            skip_acc = skip_acc + skip

        h = F.relu(self.skip_proj(F.relu(skip_acc)))  # (B, d_end, N, S)
        h = self.stformer(h)                          # (B, d_end, N, S)
        h = h[..., -4:].mean(dim=-1, keepdim=True)   # (B, d_end, N, 1)
        out = self.out_conv(h).squeeze(-1)            # (B, pred_len, N)
        return out


def build_model(cfg, A_dist_np, device):
    set_seed()
    A_fwd = torch.from_numpy(A_dist_np  ).float().to(device)
    A_bwd = torch.from_numpy(A_dist_np.T).float().to(device)
    A_fwd = A_fwd / (A_fwd.sum(1, keepdim=True) + 1e-8)
    A_bwd = A_bwd / (A_bwd.sum(1, keepdim=True) + 1e-8)
    model = MDGWNet(cfg, A_fwd, A_bwd).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'MD-GWNet v21 (fixed) | {n_params:,} parameters')
    # Smoke test
    with torch.no_grad():
        B = 2
        dummy = lambda: torch.randn(B, cfg.seq_len, cfg.num_nodes, cfg.in_features).to(device)
        tod = torch.randint(0, 288, (B, cfg.seq_len)).to(device)
        dow = torch.randint(0, 7,   (B, cfg.seq_len)).to(device)
        out = model(dummy(), dummy(), dummy(), tod, dow)
        ok  = list(out.shape) == [B, cfg.pred_len, cfg.num_nodes]
        print(f'Smoke test: {out.shape}  {chr(10003) if ok else "FAIL"}')
    return model

print('MDGWNet defined.')

In [ ]:
dl_train, dl_val, dl_test, mean_np, std_np, A_dist_np = build_dataloaders(cfg)
mean_flow = torch.from_numpy(mean_np[:, cfg.feature_idx]).to(device)
std_flow  = torch.from_numpy(std_np [:, cfg.feature_idx]).to(device)
print(f'Batches: Train={len(dl_train)} Val={len(dl_val)} Test={len(dl_test)}')

model = build_model(cfg, A_dist_np, device)

In [ ]:
def masked_huber(pred, target, delta=0.2):
    diff = torch.abs(pred - target)
    loss = torch.where(diff <= delta, 0.5*diff**2, delta*(diff - 0.5*delta))
    return loss.mean()

def masked_mae(pred, target):
    mask = (target.abs() > 0.0).float()
    return (torch.abs(pred - target) * mask).sum() / (mask.sum() + 1e-8)

def masked_rmse(pred, target):
    mask = (target.abs() > 0.0).float()
    return (((pred-target)**2 * mask).sum() / (mask.sum() + 1e-8)).sqrt()

def masked_mape(pred, target, mask_val=10.0):
    # FIX: was +1.0 (deflates MAPE) — now +1e-8 (accurate)
    mask = (target.abs() > mask_val).float()
    if mask.sum() < 1: return torch.tensor(0.0, device=pred.device)
    return (torch.abs((pred-target)/(target.abs()+1e-8)) * mask).sum() / mask.sum() * 100

print('Metrics defined (MAPE denominator fixed: +1.0 -> +1e-8).')

In [ ]:
scaler = torch.amp.GradScaler('cuda')

def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total = 0.0
    for x_rec, x_hour, x_day, y, tod, dow in loader:
        x_rec  = x_rec.to(device,  non_blocking=True)
        x_hour = x_hour.to(device, non_blocking=True)
        x_day  = x_day.to(device,  non_blocking=True)
        y      = y.to(device,      non_blocking=True)
        tod    = tod.to(device,    non_blocking=True)
        dow    = dow.to(device,    non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            pred = model(x_rec, x_hour, x_day, tod, dow)
            loss = masked_huber(pred, y, delta=cfg.huber_delta)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()   # per-batch
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, device, mean_flow, std_flow):
    model.eval()
    maes, rmses, mapes = [], [], []
    for x_rec, x_hour, x_day, y, tod, dow in loader:
        x_rec  = x_rec.to(device,  non_blocking=True)
        x_hour = x_hour.to(device, non_blocking=True)
        x_day  = x_day.to(device,  non_blocking=True)
        y      = y.to(device,      non_blocking=True)
        tod    = tod.to(device,    non_blocking=True)
        dow    = dow.to(device,    non_blocking=True)
        with torch.amp.autocast('cuda'):
            pred = model(x_rec, x_hour, x_day, tod, dow)
        pd_ = pred.float() * std_flow[None,None,:] + mean_flow[None,None,:]
        yd_ = y.float()    * std_flow[None,None,:] + mean_flow[None,None,:]
        maes.append (masked_mae (pd_, yd_).item())
        rmses.append(masked_rmse(pd_, yd_).item())
        mapes.append(masked_mape(pd_, yd_).item())
    return np.mean(maes), np.mean(rmses), np.mean(mapes)

print('Train / eval ready.')

In [ ]:
set_seed()
model = build_model(cfg, A_dist_np, device)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

# ═══════════════════════════════════════════════════════════════
# FIX: Single OneCycleLR — replaces the 3-scheduler mess
# Original had: LambdaLR warmup + CosineAnnealingWarmRestarts(T_0=100)
#               + ReduceLROnPlateau — all fighting each other
# CosineAnnealingWarmRestarts would restart LR at ep100 → val MAE
# bounce → patience=30 triggers → early stop around ep130.
# OneCycleLR: clean warmup + smooth cosine, no restarts.
# ═══════════════════════════════════════════════════════════════
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr           = cfg.lr,
    epochs           = cfg.epochs,
    steps_per_epoch  = len(dl_train),
    pct_start        = 0.08,
    anneal_strategy  = 'cos',
    div_factor       = 10.0,
    final_div_factor = 1000.0
)

best_val_mae  = float('inf')
best_val_rmse = float('inf')
best_val_mape = float('inf')
patience_cnt  = 0
history       = {'train_loss':[], 'val_mae':[], 'val_rmse':[], 'val_mape':[]}

params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'MD-GWNet v21 (fully fixed) | {params:,} params')
print(f'OneCycleLR max_lr={cfg.lr} | epochs={cfg.epochs} | patience={cfg.patience}')
print(f'Architecture: 3-stream MAF + {cfg.n_layers}-block WaveNet + STFormer(L={cfg.n_stf_layers})')
print(f'Baseline -> MAE=13.114 | RMSE=22.623 | MAPE=8.471%')
print('='*70)

for epoch in range(1, cfg.epochs+1):
    tr_loss = train_epoch(model, dl_train, optimizer, scheduler, device)
    val_mae, val_rmse, val_mape = eval_epoch(
        model, dl_val, device, mean_flow, std_flow)

    history['train_loss'].append(tr_loss)
    history['val_mae'].append(val_mae)
    history['val_rmse'].append(val_rmse)
    history['val_mape'].append(val_mape)

    tag = ''
    if val_mae < best_val_mae:
        best_val_mae  = val_mae
        best_val_rmse = val_rmse
        best_val_mape = val_mape
        patience_cnt  = 0
        torch.save(model.state_dict(), cfg.best_path)
        print(f'  Best saved -> {cfg.best_path}')
        tag = '  <- best'
    else:
        patience_cnt += 1
        if patience_cnt >= cfg.patience:
            print(f'Early stop at epoch {epoch}.')
            break

    lr_now = optimizer.param_groups[0]['lr']
    beat   = ' *** BEAT BASELINE!' if val_mae < 13.114 else ''
    if epoch % 5 == 0 or tag:
        print(f'Ep {epoch:03d} | Loss={tr_loss:.4f} | '
              f'MAE={val_mae:.3f} RMSE={val_rmse:.3f} MAPE={val_mape:.2f}% '
              f'lr={lr_now:.1e}{tag}{beat}')

print(f'\nBest Val: MAE={best_val_mae:.3f} RMSE={best_val_rmse:.3f} MAPE={best_val_mape:.2f}%')
print(f'Baseline: MAE=13.114   RMSE=22.623   MAPE=8.471%')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history['train_loss'], color='steelblue')
axes[0].set_title('Train Loss (Huber)'); axes[0].set_xlabel('Epoch')
axes[1].plot(history['val_mae'], color='tab:orange', label='Val MAE')
axes[1].axhline(13.114, color='red', ls='--', label='Baseline 13.114')
axes[1].set_title('Validation MAE'); axes[1].set_xlabel('Epoch'); axes[1].legend()
axes[2].plot(history['val_rmse'], color='tab:green', label='Val RMSE')
axes[2].axhline(22.623, color='red', ls='--', label='Baseline 22.623')
axes[2].set_title('Validation RMSE'); axes[2].set_xlabel('Epoch'); axes[2].legend()
plt.tight_layout(); plt.savefig('training_curves_v21.png', dpi=150); plt.show()

In [ ]:
model.load_state_dict(torch.load(cfg.best_path, map_location=device))

@torch.no_grad()
def paper_eval(model, loader, device, mean_flow, std_flow):
    model.eval()
    all_pred, all_true = [], []
    for x_rec, x_hour, x_day, y, tod, dow in loader:
        x_rec  = x_rec.to(device);   x_hour = x_hour.to(device)
        x_day  = x_day.to(device);   y      = y.to(device)   # FIX: y moved to device
        tod    = tod.to(device);      dow    = dow.to(device)
        with torch.amp.autocast('cuda'):
            pred = model(x_rec, x_hour, x_day, tod, dow)
        pd_ = pred.float() * std_flow[None,None,:] + mean_flow[None,None,:]
        yd_ = y.float()    * std_flow[None,None,:] + mean_flow[None,None,:]
        all_pred.append(pd_.cpu()); all_true.append(yd_.cpu())

    P = torch.cat(all_pred); Y = torch.cat(all_true)
    mae  = torch.abs(P-Y).mean().item()
    rmse = ((P-Y)**2).mean().sqrt().item()
    mask = Y.abs() > 10.0
    mape = (torch.abs((P[mask]-Y[mask])/(Y[mask].abs()+1e-8))).mean().item()*100

    print('\n' + '='*60)
    print('  TEST RESULTS — MD-GWNet v21 — all 12 steps')
    print('='*60)
    print(f'  MAE  : {mae:.3f}   baseline: 13.114   d={mae-13.114:+.3f}')
    print(f'  RMSE : {rmse:.3f}   baseline: 22.623   d={rmse-22.623:+.3f}')
    print(f'  MAPE : {mape:.3f}%  baseline:  8.471%  d={mape-8.471:+.3f}%')
    beaten = (mae < 13.114) and (rmse < 22.623) and (mape < 8.471)
    print(f'\n  Baseline BEATEN: {"YES" if beaten else "partial"}')
    print('='*60)
    return mae, rmse, mape

paper_eval(model, dl_test, device, mean_flow, std_flow)

In [ ]:
@torch.no_grad()
def horizon_eval(model, loader, device, mean_flow, std_flow):
    model.eval()
    buf = {h:{'mae':[],'rmse':[],'mape':[]} for h in [2,5,11]}
    for x_rec, x_hour, x_day, y, tod, dow in loader:
        x_rec  = x_rec.to(device);   x_hour = x_hour.to(device)
        x_day  = x_day.to(device);   y      = y.to(device)
        tod    = tod.to(device);      dow    = dow.to(device)
        with torch.amp.autocast('cuda'):
            pred = model(x_rec, x_hour, x_day, tod, dow)
        pd_ = pred.float()*std_flow[None,None,:]+mean_flow[None,None,:]
        yd_ = y.float()   *std_flow[None,None,:]+mean_flow[None,None,:]
        for h in buf:
            buf[h]['mae'].append (masked_mae (pd_[:,h,:], yd_[:,h,:]).item())
            buf[h]['rmse'].append(masked_rmse(pd_[:,h,:], yd_[:,h,:]).item())
            buf[h]['mape'].append(masked_mape(pd_[:,h,:], yd_[:,h,:]).item())
    print(f"\n{'Horizon':>14} | {'MAE':>8} | {'RMSE':>8} | {'MAPE':>9}")
    print('-'*52)
    for h, lbl in zip([2,5,11], ['3-step(15m)','6-step(30m)','12-step(60m)']):
        m = {k: np.mean(v) for k,v in buf[h].items()}
        print(f"{lbl:>14} | {m['mae']:>8.3f} | {m['rmse']:>8.3f} | {m['mape']:>8.2f}%")

horizon_eval(model, dl_test, device, mean_flow, std_flow)